### Approaches:

#### Perform hyperparameter optimization.

- Standard XGBoost Model.

#### Custom objective function.

- Standard XGBoost Model with re-focused objective function.
- Re-fitting model with weighted objective function.

#### Get accuracy and compare with ensemble models.

- Compare the expected score for each model.
- Give some theoeretical background for which model is expected to have the best score.

The model we would be using for predictions is XGBoost. Considering this as our base model, we define a hyperparameter search space.

base_model used earlier:

```python
XGBClassifier(
    objective = "multi:softprob",
    num_class = len(np.unique(train_y)),
    n_estimators = 2000,
    learning_rate = 0.05,
    max_depth = 12,
    colsmaple_bytree = 0.467,
    early_stopping_rounds = 100,
    reg_alpha = 2.7,
    reg_lambda = 1.4,
    gamma = 0.26,
    enable_categorical = True,
    tree_method = 'hist',
    max_delta_step = 4,
    subsample = 0.86,
    random_state = 13,
    device = "cuda"
)
```

so, the search space would be defined as:

```python
param_space = {
    'learning_rate':  (1e-2, 1e-1, 'log-uniform'),
    'max_depth':      (4, 14),
    'min_child_weight': (1, 7),
    'gamma':          (0, 0.5),
    'subsample':      (0.6, 1.0),
    'colsample_bytree': (0.4, 1.0),
    'reg_alpha':      (0, 5, 'log-uniform'),
    'reg_lambda':     (0.5, 5, 'log-uniform'),
    'max_delta_step': (0, 5),
}
```

In [85]:
from scipy.stats import uniform, reciprocal, randint

def sample_param(distribution):
    low, high = distribution[:2]
    
    if len(distribution) == 3:
        kind = distribution[2]
        if kind == 'log-uniform':
            return reciprocal(low, high).rvs()
        elif kind == 'integer':
            return randint(low, high + 1).rvs()
    
    # Default: uniform
    return uniform(loc=low, scale=high - low).rvs()


In [87]:
sample_param((0.3, 1.0))

np.float64(0.5529978352949136)

## Just initial feature-stuff
- do pre-process to remove unnecessary stuff.
- encode categorical features.
- separate our training and test sets.
- create features defined earlier for the training set (with proper pipeline in the case of test set.)

In [11]:
import pandas as pd
import numpy as np
import glob

In [18]:
files = glob.glob("../data/*/*")
files

['../data/s5e6/test.csv',
 '../data/s5e6/previous_competition.csv',
 '../data/s5e6/train.csv',
 '../data/s5e6/sample_submission.csv']

In [19]:
train_df = pd.read_csv(files[2]).drop(columns = ["id"])
test_df = pd.read_csv(files[0]).drop(columns = ["id"])
original_df = pd.read_csv(files[1])
submission_df = pd.read_csv(files[3])

In [20]:
train_df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,27,69,65,Sandy,Millets,30,6,18,28-28
2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,35,58,43,Red,Paddy,37,2,16,DAP


In [21]:
target_feature = list(train_df.columns)[-1]
numerical_features = [x for x in train_df.describe().columns if x != "id"]
categorical_features = [x for x in train_df.columns if x not in numerical_features and x != "id" and x != target_feature]
numerical_features, categorical_features

(['Temparature',
  'Humidity',
  'Moisture',
  'Nitrogen',
  'Potassium',
  'Phosphorous'],
 ['Soil Type', 'Crop Type'])

In [23]:
train_df = pd.concat([train_df, original_df], ignore_index = True)

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from xgboost import XGBClassifier

In [25]:
oe = OrdinalEncoder()
train_df[categorical_features] = oe.fit_transform(train_df[categorical_features])
for col in categorical_features:
    train_df[col] = train_df[col].astype("category")
le = LabelEncoder()
train_df[target_feature] = le.fit_transform(train_df[target_feature])
train_df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,37,70,36,1.0,8.0,36,4,5,4
1,27,69,65,4.0,4.0,30,6,18,4
2,29,63,32,4.0,4.0,24,12,16,2
3,35,62,54,4.0,0.0,39,12,4,0
4,35,58,43,3.0,6.0,37,2,16,5


**separate train and validation set (validation as in the set which would be used not be used for scaling for now)**
- validation set is not used for scaling when normalizing for doing feature engineering.

In [38]:
full_y, full_x = train_df[target_feature], train_df.drop(columns = [target_feature])
# train_x, test_x, train_y, test_y = train_test_split(full_x, full_y, test_size = 0.1, random_state = 42, stratify = full_y) ## keeping 10% of the data as test data for now.

### Feature engineering:
- Binning temperature, humidity.
- adding temperature and humidity (after min-max normalization)
- dividing humidity in soil by humidity in air.
- add all normalized mineral values (nitrogen, phosphorous & pottasium)

In [39]:
full_x.columns

Index(['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous'],
      dtype='object')

In [40]:
temp_feature, humid_feature, moist_feature = full_x.columns[:3]
mineral_features = full_x.columns[-3:]
mineral_features

Index(['Nitrogen', 'Potassium', 'Phosphorous'], dtype='object')

In [61]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler, MinMaxScaler

class TempHumidityIndexAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Assumes X has two columns: temperature and humidity
        return (X[:, 0] + X[:, 1]).reshape(-1, 1)

class HumidityMoistureRatioAdder(BaseEstimator, TransformerMixin):
    def __init__(self, epsilon=1e-6):
        self.epsilon = epsilon
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        # X has shape (n_samples, 2): [humidity, moisture]
        ratio = X[:, 0] / (X[:, 1] + self.epsilon)
        return ratio.reshape(-1, 1)

class SummarizedMineralAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.scaler = MinMaxScaler()
        self.scaler.fit(X)
        return self
    def transform(self, X):
        nmv = self.scaler.transform(X)
        return np.sum(nmv, axis=1).reshape(-1, 1)

class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        q1 = np.percentile(X, 25, axis=0)
        q3 = np.percentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower = q1 - 1.5 * iqr
        self.upper = q3 + 1.5 * iqr
        return self
    def transform(self, X):
        return np.clip(X, self.lower, self.upper)

## binning temperature, humidity.
binning_pipeline = Pipeline(steps=[
    ('discretizer', KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile'))
])

## adding temperature, humidity.
temp_hum_index_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('index_adder', TempHumidityIndexAdder())
])

# Humidity / Moisture Ratio
humidity_moisture_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ratio', HumidityMoistureRatioAdder()),
    ('clip', IQRClipper())
])

# Summarized Mineral Values
summarized_mineral_pipeline = Pipeline([
    ('adder', SummarizedMineralAdder()),
    ('clip', IQRClipper())
])

preprocessor = ColumnTransformer([
    ('binning', binning_pipeline, [temp_feature, humid_feature]),
    ('temp_hum_index', temp_hum_index_pipeline, [temp_feature, humid_feature]),
    ('humidity_moisture_ratio', humidity_moisture_pipeline, [humid_feature, temp_feature]),
    ('summarized_minerals', summarized_mineral_pipeline, mineral_features)
])

def create_preprocessed_model_pipeline(base_model):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', base_model)
    ])

#### Three different base-models
- Vanilla XGBoost.
- XGBoost with smooth surrogate-loss function.
- Ensemble binary loss XGBoost.

In [62]:
from scipy.special import expit

def smooth_map3_obj(preds, dtrain):
    """
    preds:  flat array of length N*K (raw scores)
    dtrain: DMatrix containing labels in [0..K-1]
    Returns (grad, hess) both flat arrays of same length
    """
    labels = dtrain.get_label().astype(int)
    N = labels.shape[0]
    K = int(preds.size / N)
    # reshape to (N, K)
    S = preds.reshape(N, K)
    
    grad = np.zeros_like(S)
    hess = np.zeros_like(S)
    
    # for each sample
    for i in range(N):
        y = labels[i]
        scores = S[i]
        # get the three largest *other* scores
        others = np.delete(scores, y)
        top3_idx = np.argsort(others)[-3:][::-1]  # indices in 'others'
        top3_scores = others[top3_idx]
        
        # build the surrogate:  L = -[ σ(s_y − t₁)
        #                         + (1/2) σ(s_y − t₂)
        #                         + (1/3) σ(s_y − t₃) ]
        diffs = scores[y] - top3_scores  # shape (≤3,)
        weights = np.array([1.0, 0.5, 1/3.0])
        σ = expit(diffs)
        
        # gradient w.r.t. s_y:
        #   ∂L/∂s_y = −Σ_k w_k σ'(diffs_k)
        # ∂L/∂t_k = +w_k σ'(diffs_k)
        sigp = σ * (1 - σ)
        # sum over however many we have (<3 if K<4)
        grad_y = -np.sum(weights[:len(sigp)] * sigp)
        hess_y =  np.sum(weights[:len(sigp)] * sigp * (1 - 2*σ))
        
        grad[i, y] = grad_y
        hess[i, y] = hess_y
        
        # distribute to the top-3 others
        # need to map back to original class indices
        idx_map = [j for j in range(K) if j != y]
        for rank, idx_o in enumerate(top3_idx):
            j = idx_map[idx_o]
            grad[i, j] =  weights[rank] * sigp[rank]
            hess[i, j] = weights[rank] * sigp[rank] * (1 - 2*σ[rank])
    
    # flatten back out
    return grad.ravel(), hess.ravel()

In [124]:
from sklearn.base import BaseEstimator, ClassifierMixin

class XGBTrainWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, params, num_boost_round=2000, early_stopping_rounds=100, obj=None, eval_metric="mlogloss", verbose_eval=False):
        self.params = params.copy()
        self.num_boost_round = num_boost_round
        self.early_stopping_rounds = early_stopping_rounds
        self.obj = obj
        self.eval_metric = eval_metric
        self.verbose_eval = verbose_eval

    def fit(self, X, y, eval_set=None):
        # Build DMatrices
        # 1) Convert any pandas -> numpy
        if isinstance(X, (pd.DataFrame, pd.Series)):
            X_train = X.values
        else:
            X_train = X
        if isinstance(y, (pd.Series, pd.DataFrame)):
            y_train = y.values
        else:
            y_train = y

        # 2) Build DMatrix for train
        dtrain = xgb.DMatrix(X_train, label=y_train)

        # 3) Prepare eval list -- skip because too much code :(
        # evals = [(dtrain, "train")]
        # if eval_set is not None:
        #     X_val, y_val = eval_set
        #     # again, force numpy
        #     if isinstance(X_val, (pd.DataFrame, pd.Series)):
        #         X_val = X_val.values
        #     if isinstance(y_val, (pd.Series, pd.DataFrame)):
        #         y_val = y_val.values
        #     evals.append((xgb.DMatrix(X_val, label=y_val), "valid"))

        # train
        self.bst_ = xgb.train(
            self.params,
            dtrain,
            obj=self.obj,
#            evals=evals,
#            evals_result=self._evals_result if hasattr(self, "_evals_result") else {},
#            early_stopping_rounds=self.early_stopping_rounds,
            verbose_eval=self.verbose_eval,
        )
        return self

    def predict_proba(self, X):
        dtest = xgb.DMatrix(X)
        probs = self.bst_.predict(dtest)
        # xgb.train with multi:softprob outputs flat or (n,K)?
        return probs.reshape(X.shape[0], -1)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [ ]:
def create_custom_preprocessed_model_pipeline(custom_obj, all_params):
    xgb_wrapper = XGBTrainWrapper(
        params=all_params,
        early_stopping_rounds=100,
        obj=custom_obj,
        eval_metric='mlogloss',
        verbose_eval=50
    )

    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier',    xgb_wrapper)
    ])

#### Implementing Random Search CV for XGBoost Models.

In [75]:
param_space = {
    'n_estimators':      (1000, 4000, 'integer'),
    'learning_rate':     (1e-2, 1e-1, 'log-uniform'),
    'max_depth':         (4, 14, 'integer'),
    'min_child_weight':  (1, 7, 'integer'),
    'gamma':             (0, 0.5),
    'subsample':         (0.6, 1.0),
    'colsample_bytree':  (0.4, 1.0),
    'reg_alpha':         (1e-3, 5, 'log-uniform'), 
    'reg_lambda':        (0.5, 5, 'log-uniform'),
    'max_delta_step':    (0, 5, 'integer'),
}


In [41]:
X, y = full_x, full_y

In [64]:
full_x.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,37,70,36,1.0,8.0,36,4,5
1,27,69,65,4.0,4.0,30,6,18
2,29,63,32,4.0,4.0,24,12,16
3,35,62,54,4.0,0.0,39,12,4
4,35,58,43,3.0,6.0,37,2,16


In [65]:
full_y.head()

0    4
1    4
2    2
3    0
4    5
Name: Fertilizer Name, dtype: int64

In [66]:
import xgboost as xgb

In [93]:
def map3_score(model, x_test, y_test):
    y_pred_probs = model.predict_proba(x_test)
    top3_probs = np.argsort(y_pred_probs, axis = 1)[:, -3:][:, ::-1]
    return get_best_and_full_accuracy_xgboost(y_test, top3_probs)

def _get_score(actual, predicted):
    score = 0.0
    hits = 0
    seen = set()
    for i, pred in enumerate(predicted):
        if pred == np.int64(actual) and pred not in seen:
            hits += 1
            score += hits / (i + 1.0)
            seen.add(pred)
    
    return score ## since actual is ONE entity.


def get_best_and_full_accuracy_xgboost(ptest_y, topk_probs):
    test_yl = ptest_y.tolist()
    first_acc_l = [x for idx, x in enumerate(test_yl) if np.int64(x) == topk_probs[idx][0]]
    score_accl_l = [_get_score(x ,topk_probs[idx]) for idx, x in enumerate(test_yl)]

    return np.mean(score_accl_l)

In [120]:
from sklearn.model_selection import StratifiedKFold as skfold

def process_experiment(total_exps = 10, is_vanilla = True):
    best_params = None
    best_model = None
    best_score = 0
    param_score_dictionary = {}

    kf = skfold(n_splits = 5, shuffle = True, random_state = 42)

    for _ in range(total_exps):
        test_scores = []
        best_rounds = []
        params = {k: sample_param(v) for k, v in param_space.items()}

        for train_index, test_index in kf.split(X, y):
            X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
            y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

            # Further split current train set into train and validation 
            X_train_fold, X_val, y_train_fold, y_val = train_test_split(X_train_fold, y_train_fold, test_size=0.2, random_state=42)

            all_params = {
                "objective": "multi:softprob",
                "num_class": len(np.unique(y)),
                "enable_categorical": True,
                "random_state": 13,
            }
            all_params.update(params)

            if is_vanilla:
                # base_model if vanilla XGBoost.
                base_model = xgb.XGBClassifier(**all_params)
                pipeline_model = create_preprocessed_model_pipeline(base_model)   
                pipeline_model.fit(X_train_fold, y_train_fold) 
            else:
                other_params = {
                    "early_stopping_rounds": 100,
                    "verbose_eval": 50
                }
                all_params.update(other_params)
                pipeline_model = create_custom_preprocessed_model_pipeline(smooth_map3_obj, all_params)
                X_val_np, y_val_np = X_val.values, y_val.values
                pipeline_model.fit(X_train_fold, y_train_fold, classifier__eval_set = (X_val, y_val))


            test_score = map3_score(pipeline_model, X_test_fold, y_test_fold)
            test_scores.append(test_score)


        ## average score across all folds.
        average_score = np.mean(test_scores)
        if average_score > best_score:
            best_score = average_score
            best_params = params
        param_score_dictionary[str(params)] = average_score

    print(f"best parameters: {best_params}")
    print(f"best score: {best_score}")

    return param_score_dictionary

**without the custom objective**

In [97]:
vanilla_param_score_dict = process_experiment() 

/Users/abhishekmish/Documents/repos/kaggle_playground/.venv/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/abhishekmish/Documents/repos/kaggle_playground/.venv/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/abhishekmish/Documents/repos/kaggle_playground/.venv/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 in

best parameters: {'n_estimators': 1003, 'learning_rate': np.float64(0.05342108208004262), 'max_depth': 6, 'min_child_weight': 5, 'gamma': np.float64(0.48502506052657185), 'subsample': np.float64(0.96080877356403), 'colsample_bytree': np.float64(0.49691387116467284), 'reg_alpha': np.float64(0.7083290168153822), 'reg_lambda': np.float64(3.099158582401177), 'max_delta_step': 3}
best score: 0.28634313725490196


In [98]:
import pickle
with open('vanilla_param_score_dict.pkl', 'wb') as f:
    pickle.dump(vanilla_param_score_dict, f)

**with the custom objective**

In [125]:
optimized_param_score_dict = process_experiment(is_vanilla = False)

/Users/abhishekmish/Documents/repos/kaggle_playground/.venv/lib/python3.12/site-packages/xgboost/core.py:729: UserWarning: [22:01:14] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "early_stopping_rounds", "enable_categorical", "n_estimators", "verbose_eval" } are not used.

  return func(**kwargs)
/Users/abhishekmish/Documents/repos/kaggle_playground/.venv/lib/python3.12/site-packages/xgboost/core.py:2291: FutureWarning: Since 2.1.0, the shape of the gradient and hessian is required to be (n_samples, n_targets) or (n_samples, n_classes).
  warnings.warn(
/Users/abhishekmish/Documents/repos/kaggle_playground/.venv/lib/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/abhishekmish/Documents/repos/kaggle_playground/.ve

best parameters: {'n_estimators': 1081, 'learning_rate': np.float64(0.04290976064460291), 'max_depth': 7, 'min_child_weight': 2, 'gamma': np.float64(0.3712127835010872), 'subsample': np.float64(0.7664822627304613), 'colsample_bytree': np.float64(0.7829028534049483), 'reg_alpha': np.float64(0.014274306240603633), 'reg_lambda': np.float64(1.9029194745369447), 'max_delta_step': 1}
best score: 0.23879843137254902


In [126]:
with open('optimized_param_score_dict.pkl', 'wb') as f:
    pickle.dump(optimized_param_score_dict, f)

**binary ensemble log-loss pipeline**

#### Estimates of performance.